# Tier C: The Transformer
## Parameter-Efficient Fine-Tuning with LoRA for Stylometry Detection

**Objective:** Use state-of-the-art transformer models with PEFT (LoRA) for Human vs AI text classification

**Architecture:**
- **Base Model**: DistilBERT (distilbert-base-uncased)
- **Fine-Tuning Method**: LoRA (Low-Rank Adaptation)
- **Task**: Binary sequence classification

---

## 🔗 Building on Task 1 & Task 2 (Tiers A & B)

**Evolution of Approaches:**

### **Task 1: The Explorer** (Exploratory Data Analysis)
- **Focus**: Mathematical proof of distinction
- **Findings**: TTR variance, structural drift, POS entropy differentiate Human/AI
- **Result**: Established statistical baselines

### **Task 2.1 (Tier A): The Statistician** (Traditional ML)
- **Focus**: Hand-crafted features + XGBoost
- **Method**: Engineer features based on Task 1 discoveries
- **Strength**: Interpretable, fast, baseline performance

### **Task 2.2 (Tier B): The Semanticist** (Neural + Embeddings)
- **Focus**: Pre-trained semantics + feedforward network
- **Method**: GloVe embeddings capture word meaning
- **Strength**: Semantic relationships, better than pure statistics

### **Task 2.3 (Tier C): The Transformer** (SOTA + PEFT)
- **Focus**: Contextual embeddings + self-attention
- **Method**: DistilBERT with LoRA fine-tuning
- **Strength**: Captures long-range dependencies, contextual understanding
- **Expected**: Best performance, learns patterns Task 1 discovered + more

**Why This Progression Matters:**
1. Task 1 proved mathematical distinction exists
2. Tier A validated we can classify with interpretable features
3. Tier B showed semantic information helps
4. Tier C tests if transformers can **automatically discover** what Task 1 found manually

---

**Key Features:**
- **Parameter Efficient**: Train only ~0.6% of model parameters
- **LoRA Configuration**: r=8, alpha=16, dropout=0.1
- **Target Modules**: Query and Value attention layers
- **Evaluation**: Overall accuracy + Mimic subset analysis

---

**Why LoRA?**
- Reduces trainable parameters by 99%+
- Maintains performance comparable to full fine-tuning
- Faster training and lower memory requirements
- Enables fine-tuning large models on consumer GPUs
- **Can learn patterns from Task 1 without explicit feature engineering**

## 1. Setup and Installation

In [7]:
# Install required packages
!pip install transformers datasets peft accelerate scikit-learn pandas numpy torch -q

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


In [8]:
# Import libraries
import pandas as pd
import numpy as np
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

# Hugging Face libraries
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# PEFT (Parameter-Efficient Fine-Tuning)
from peft import LoraConfig, get_peft_model, TaskType

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set random seeds
import torch
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

✅ Libraries imported successfully!
PyTorch version: 2.8.0+cu128
CUDA available: False


## 2. Configuration

**IMPORTANT:** Update these file paths to match your CSV locations

In [9]:
# CSV file paths
CSV_PATHS = {
    "Class 1 (Human)": {
        "path": "/home/avani/precog/content/precog.csv",
        "text_column": "text",
        "label": "Human"
    },
    "Class 2 (AI)": {
        "path": "/home/avani/precog/content/class_2_pro_vanilla_combined.csv",
        "text_column": "text",
        "label_column": "author_target"
    },
    "Class 3 (AI Mimic)": {
        "path": "/home/avani/precog/content/class_3_pro_combined.csv",
        "text_column": "text",
        "label_column": "author_target"
    }
}

# Model configuration
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512
NUM_LABELS = 2

# LoRA configuration
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_lin", "v_lin"]  # DistilBERT attention layers

# Training configuration
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 16
TEST_SIZE = 0.2

# Output directory
OUTPUT_DIR = "/home/avani/precog/reports/lora_distilbert"

print("✅ Configuration complete!")
print(f"\n📊 Model: {MODEL_NAME}")
print(f"📊 LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"📊 Target modules: {TARGET_MODULES}")
print(f"📊 Training: {NUM_EPOCHS} epochs, LR={LEARNING_RATE}")

✅ Configuration complete!

📊 Model: distilbert-base-uncased
📊 LoRA r=8, alpha=16, dropout=0.1
📊 Target modules: ['q_lin', 'v_lin']
📊 Training: 2 epochs, LR=0.0002


## 3. Load and Prepare Data

In [10]:
def load_csv_files() -> pd.DataFrame:
    """
    Load and combine all three CSV files.
    
    Returns:
        Combined DataFrame with 'text' and 'label' columns
    """
    print("=" * 70)
    print("LOADING CSV FILES")
    print("=" * 70)
    
    all_dfs = []
    
    for class_name, config in CSV_PATHS.items():
        try:
            print(f"\n📂 Loading {class_name}: {config['path']}")
            df = pd.read_csv(config['path'])
            
            # Set label
            if 'label_column' in config and config['label_column'] in df.columns:
                df['label'] = df[config['label_column']]
            else:
                df['label'] = config.get('label', 'Unknown')
            
            # Keep only text and label
            df = df[[config['text_column'], 'label']].copy()
            df.columns = ['text', 'label']
            
            print(f"   ✓ Loaded {len(df)} samples")
            all_dfs.append(df)
            
        except FileNotFoundError:
            print(f"   ✗ File not found: {config['path']}")
        except Exception as e:
            print(f"   ✗ Error: {e}")
    
    if not all_dfs:
        raise FileNotFoundError("No CSV files could be loaded.")
    
    df_combined = pd.concat(all_dfs, ignore_index=True)
    
    print(f"\n{'=' * 70}")
    print(f"COMBINED DATASET: {len(df_combined)} total samples")
    print(f"{'=' * 70}")
    print(f"\nLabel distribution:")
    print(df_combined['label'].value_counts())
    
    return df_combined


# Load data
df = load_csv_files()

LOADING CSV FILES

📂 Loading Class 1 (Human): /home/avani/precog/content/precog.csv
   ✓ Loaded 2492 samples

📂 Loading Class 2 (AI): /home/avani/precog/content/class_2_pro_vanilla_combined.csv
   ✓ Loaded 500 samples

📂 Loading Class 3 (AI Mimic): /home/avani/precog/content/class_3_pro_combined.csv
   ✓ Loaded 500 samples

COMBINED DATASET: 3492 total samples

Label distribution:
label
Human                     2492
AI_Generic                 500
Robert Louis Stevenson     250
Arthur Conan Doyle         250
Name: count, dtype: int64


In [11]:
# Create binary labels
print("\n=" * 70)
print("CREATING BINARY LABELS")
print("=" * 70)

df['binary_label'] = df['label'].apply(lambda x: 0 if x.lower() == 'human' else 1)

print(f"\nBinary label distribution:")
print(f"  Human (0): {(df['binary_label'] == 0).sum()} samples")
print(f"  AI (1):    {(df['binary_label'] == 1).sum()} samples")

# Display sample
print(f"\n📝 Sample data:")
print(df[['text', 'label', 'binary_label']].head(3))


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
CREATING BINARY LABELS

Binary label distribution:
  Human (0): 2492 samples
  AI (1):    1000 samples

📝 Sample data:
                                                text  label  binary_label
0  The Strange Case of Dr. Jekyll and Mr. Hyde by...  Human             0
1  Mr. Utterson the lawyer was a man of a rugged ...  Human             0
2  No doubt the feat was easy to Mr. Utterson; fo...  Human             0


## 4. Convert to Hugging Face Dataset

In [12]:
from sklearn.model_selection import train_test_split

print("=" * 70)
print("CREATING HUGGING FACE DATASET")
print("=" * 70)

# Train-test split
train_df, test_df = train_test_split(
    df[['text', 'label', 'binary_label']], 
    test_size=TEST_SIZE, 
    random_state=SEED, 
    stratify=df['binary_label']
)

print(f"\n✓ Train set: {len(train_df)} samples")
print(f"✓ Test set:  {len(test_df)} samples")

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df[['text', 'binary_label']].rename(columns={'binary_label': 'labels'}))
test_dataset = Dataset.from_pandas(test_df[['text', 'binary_label']].rename(columns={'binary_label': 'labels'}))

# Store original labels for mimic analysis
test_labels_original = test_df['label'].values

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print(f"\n✅ Hugging Face Dataset created:")
print(dataset)

CREATING HUGGING FACE DATASET

✓ Train set: 2793 samples
✓ Test set:  699 samples

✅ Hugging Face Dataset created:
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__'],
        num_rows: 2793
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__'],
        num_rows: 699
    })
})


## 5. Load Tokenizer and Tokenize Data

In [13]:
print("=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"\n✓ Loaded tokenizer: {MODEL_NAME}")
print(f"  Vocabulary size: {len(tokenizer)}")
print(f"  Max length: {MAX_LENGTH}")

# Test tokenizer
test_text = "The quick brown fox jumps over the lazy dog."
test_tokens = tokenizer(test_text, truncation=True, max_length=MAX_LENGTH)
print(f"\n📝 Test tokenization:")
print(f"  Text: {test_text}")
print(f"  Tokens: {tokenizer.convert_ids_to_tokens(test_tokens['input_ids'][:10])}...")
print(f"  Token IDs: {test_tokens['input_ids'][:10]}...")

LOADING TOKENIZER

✓ Loaded tokenizer: distilbert-base-uncased
  Vocabulary size: 30522
  Max length: 512

📝 Test tokenization:
  Text: The quick brown fox jumps over the lazy dog.
  Tokens: ['[CLS]', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']...
  Token IDs: [101, 1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899]...

✓ Loaded tokenizer: distilbert-base-uncased
  Vocabulary size: 30522
  Max length: 512

📝 Test tokenization:
  Text: The quick brown fox jumps over the lazy dog.
  Tokens: ['[CLS]', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']...
  Token IDs: [101, 1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899]...


In [14]:
def tokenize_function(examples):
    """
    Tokenize text data.
    """
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False  # Will pad dynamically during training
    )


print("\n=" * 70)
print("TOKENIZING DATASET")
print("=" * 70)
print("\n⚙️  Tokenizing texts (this may take a moment...)\n")

# Tokenize datasets
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    desc="Tokenizing"
)

print(f"\n✅ Tokenization complete:")
print(tokenized_datasets)

# Show sample
print(f"\n📝 Sample tokenized data:")
print(f"  Keys: {tokenized_datasets['train'][0].keys()}")
print(f"  Input IDs length: {len(tokenized_datasets['train'][0]['input_ids'])}")
print(f"  Label: {tokenized_datasets['train'][0]['labels']}")


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
TOKENIZING DATASET

⚙️  Tokenizing texts (this may take a moment...)



Tokenizing:   0%|          | 0/2793 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/699 [00:00<?, ? examples/s]


✅ Tokenization complete:
DatasetDict({
    train: Dataset({
        features: ['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2793
    })
    test: Dataset({
        features: ['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 699
    })
})

📝 Sample tokenized data:
  Keys: dict_keys(['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])
  Input IDs length: 155
  Label: 0


In [15]:
# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("✅ Data collator created for dynamic padding")

✅ Data collator created for dynamic padding


## 6. Load Base Model

In [16]:
print("=" * 70)
print("LOADING BASE MODEL")
print("=" * 70)

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={0: "Human", 1: "AI"},
    label2id={"Human": 0, "AI": 1}
)

print(f"\n✓ Loaded model: {MODEL_NAME}")
print(f"  Task: Sequence Classification")
print(f"  Number of labels: {NUM_LABELS}")

# Count parameters before LoRA
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model parameters (BEFORE LoRA):")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

LOADING BASE MODEL


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✓ Loaded model: distilbert-base-uncased
  Task: Sequence Classification
  Number of labels: 2

📊 Model parameters (BEFORE LoRA):
  Total parameters: 66,955,010
  Trainable parameters: 66,955,010


## 7. Apply LoRA (Parameter-Efficient Fine-Tuning)

**LoRA Configuration:**
- **r (rank)**: 8 - Dimension of low-rank matrices
- **alpha**: 16 - Scaling factor for LoRA updates
- **dropout**: 0.1 - Regularization
- **target_modules**: ["q_lin", "v_lin"] - Apply LoRA to Query and Value attention layers only

This reduces trainable parameters from ~67M to ~0.4M (~0.6%)

In [17]:
print("=" * 70)
print("APPLYING LoRA (LOW-RANK ADAPTATION)")
print("=" * 70)

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    inference_mode=False
)

print(f"\n⚙️  LoRA Configuration:")
print(f"  r (rank): {LORA_R}")
print(f"  alpha: {LORA_ALPHA}")
print(f"  dropout: {LORA_DROPOUT}")
print(f"  target_modules: {TARGET_MODULES}")

# Apply LoRA to model
model = get_peft_model(model, lora_config)

print(f"\n✅ LoRA applied successfully!")

APPLYING LoRA (LOW-RANK ADAPTATION)

⚙️  LoRA Configuration:
  r (rank): 8
  alpha: 16
  dropout: 0.1
  target_modules: ['q_lin', 'v_lin']

✅ LoRA applied successfully!


In [18]:
# Print trainable parameters (AFTER LoRA)
print("\n" + "=" * 70)
print("EFFICIENCY CHECK: TRAINABLE PARAMETERS")
print("=" * 70)

model.print_trainable_parameters()

# Calculate manually for clarity
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_percentage = 100 * trainable_params / total_params

print(f"\n📊 Detailed parameter breakdown:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Trainable %: {trainable_percentage:.2f}%")
print(f"\n✨ Training only {trainable_percentage:.2f}% of parameters!")
print(f"   This is {100 - trainable_percentage:.2f}% more efficient than full fine-tuning!")


EFFICIENCY CHECK: TRAINABLE PARAMETERS
trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925

📊 Detailed parameter breakdown:
  Total parameters: 67,694,596
  Trainable parameters: 739,586
  Trainable %: 1.09%

✨ Training only 1.09% of parameters!
   This is 98.91% more efficient than full fine-tuning!


## 8. Define Training Arguments

In [19]:
print("=" * 70)
print("CONFIGURING TRAINING")
print("=" * 70)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    eval_strategy="epoch",  # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    seed=SEED,
    report_to="none",  # Disable wandb/tensorboard for simplicity
    push_to_hub=False
)

print(f"\n⚙️  Training configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"\n✅ Training arguments configured")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


CONFIGURING TRAINING

⚙️  Training configuration:
  Epochs: 2
  Learning rate: 0.0002
  Batch size: 16
  Output directory: /home/avani/precog/reports/lora_distilbert

✅ Training arguments configured


## 9. Define Metrics

In [20]:
def compute_metrics(eval_pred):
    """
    Compute accuracy for evaluation.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy}


print("✅ Metrics function defined")

✅ Metrics function defined


## 10. Initialize Trainer

In [21]:
print("=" * 70)
print("INITIALIZING TRAINER")
print("=" * 70)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("\n✅ Trainer initialized successfully!")
print(f"  Training samples: {len(tokenized_datasets['train'])}")
print(f"  Evaluation samples: {len(tokenized_datasets['test'])}")


INITIALIZING TRAINER

✅ Trainer initialized successfully!
  Training samples: 2793
  Evaluation samples: 699


## 11. Train the Model

⚠️ **This will take several minutes depending on your GPU/CPU.**

In [22]:
print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print("\n🚀 Training LoRA-adapted DistilBERT...\n")

# Train
train_result = trainer.train()

print("\n" + "=" * 70)
print("✅ TRAINING COMPLETE!")
print("=" * 70)

# Print training metrics
print(f"\n📊 Training metrics:")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value:.4f}")

STARTING TRAINING

🚀 Training LoRA-adapted DistilBERT...



Epoch,Training Loss,Validation Loss,Accuracy
1,0.032332,0.015277,0.994278
2,0.005318,0.006768,0.997139



✅ TRAINING COMPLETE!

📊 Training metrics:
  train_runtime: 8407.4676
  train_samples_per_second: 0.6640
  train_steps_per_second: 0.0420
  total_flos: 546878304476592.0000
  train_loss: 0.1080
  epoch: 2.0000


## 12. Evaluate on Test Set

In [23]:
print("=" * 70)
print("EVALUATING ON TEST SET")
print("=" * 70)

# Evaluate
eval_results = trainer.evaluate()

print(f"\n📊 Test Set Performance:")
print("=" * 70)
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

test_accuracy = eval_results['eval_accuracy']
print(f"\n✨ Final Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

EVALUATING ON TEST SET



📊 Test Set Performance:
  eval_loss: 0.0068
  eval_accuracy: 0.9971
  eval_runtime: 237.5419
  eval_samples_per_second: 2.9430
  eval_steps_per_second: 0.1850
  epoch: 2.0000

✨ Final Test Accuracy: 0.9971 (99.71%)


In [24]:
# Get predictions for detailed analysis
print("\n🔮 Generating predictions for detailed analysis...\n")

predictions = trainer.predict(tokenized_datasets['test'])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print("✅ Predictions generated")


🔮 Generating predictions for detailed analysis...

✅ Predictions generated
✅ Predictions generated


## 13. Detailed Evaluation Metrics

In [ ]:
print("=" * 70)
print("DETAILED EVALUATION METRICS")
print("=" * 70)

# Classification report
print("\n📋 Classification Report:")
print("=" * 70)
print(classification_report(
    y_true, 
    y_pred, 
    target_names=['Human (0)', 'AI (1)'],
    digits=4
))

In [ ]:
# Confusion matrix
print("\n🎯 Confusion Matrix:")
print("=" * 70)
cm = confusion_matrix(y_true, y_pred)
print(f"\n           Predicted")
print(f"           Human  AI")
print(f"Actual Human  {cm[0][0]:4d}  {cm[0][1]:4d}")
print(f"       AI     {cm[1][0]:4d}  {cm[1][1]:4d}")

# Calculate additional metrics
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n📈 Additional Metrics:")
print("=" * 70)
print(f"  True Negatives:  {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  True Positives:  {tp}")
print(f"\n  Sensitivity (Recall): {sensitivity:.4f}")
print(f"  Specificity:          {specificity:.4f}")

## 14. BONUS: Mimic Subset Evaluation

Evaluate specifically on samples labeled 'Arthur Conan Doyle' or 'Robert Louis Stevenson' to see if LoRA can detect sophisticated AI mimics.

In [ ]:
print("=" * 70)
print("BONUS: MIMIC SUBSET EVALUATION")
print("=" * 70)

# Filter for mimic samples
mimic_mask = np.array([
    'Doyle' in str(label) or 'Stevenson' in str(label) 
    for label in test_labels_original
])

if mimic_mask.sum() > 0:
    y_mimic_true = y_true[mimic_mask]
    y_mimic_pred = y_pred[mimic_mask]
    mimic_labels = test_labels_original[mimic_mask]
    
    print(f"\n🎭 Found {mimic_mask.sum()} mimic samples in test set")
    print(f"\nMimic label distribution:")
    unique, counts = np.unique(mimic_labels, return_counts=True)
    for label, count in zip(unique, counts):
        print(f"  {label}: {count}")
    
    # Calculate mimic accuracy
    mimic_accuracy = accuracy_score(y_mimic_true, y_mimic_pred)
    
    print(f"\n📊 Mimic Subset Performance:")
    print("=" * 70)
    print(f"  Mimic Accuracy: {mimic_accuracy:.4f} ({mimic_accuracy*100:.2f}%)")
    
    print(f"\n📋 Mimic Classification Report:")
    print("=" * 70)
    print(classification_report(
        y_mimic_true, 
        y_mimic_pred,
        target_names=['Human (0)', 'AI (1)'],
        digits=4
    ))
    
    # Detailed breakdown by author
    print(f"\n🔍 Detailed Mimic Breakdown:")
    print("=" * 70)
    for label in unique:
        label_mask = mimic_labels == label
        label_accuracy = accuracy_score(
            y_mimic_true[label_mask], 
            y_mimic_pred[label_mask]
        )
        print(f"  {label:35s}: {label_accuracy:.4f} ({label_accuracy*100:.2f}%)")
    
    # Comparison
    print(f"\n💡 Comparison:")
    print("=" * 70)
    print(f"  Overall test accuracy: {test_accuracy:.4f}")
    print(f"  Mimic subset accuracy: {mimic_accuracy:.4f}")
    print(f"  Difference: {abs(test_accuracy - mimic_accuracy):.4f}")
    
    if mimic_accuracy < test_accuracy:
        print(f"\n⚠️  Mimics are harder to detect! (Lower accuracy by {(test_accuracy - mimic_accuracy)*100:.2f}%)")
    else:
        print(f"\n✅ Mimics are equally or more detectable!")
else:
    print("\n⚠️  No mimic samples found in test set")
    print("(Looking for labels containing 'Doyle' or 'Stevenson')")

## 15. Save Model

In [ ]:
print("=" * 70)
print("SAVING MODEL")
print("=" * 70)

# Save the LoRA adapter
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

print(f"\n💾 LoRA adapter saved to: {OUTPUT_DIR}/lora_adapter")
print(f"\n📝 Note: Only LoRA weights are saved (~1-2MB), not the full model!")
print(f"   To load: Load base model + apply saved LoRA adapter")

# Save evaluation results
results_df = pd.DataFrame({
    'metric': list(eval_results.keys()),
    'value': list(eval_results.values())
})
results_df.to_csv(f"{OUTPUT_DIR}/evaluation_results.csv", index=False)

print(f"\n💾 Evaluation results saved to: {OUTPUT_DIR}/evaluation_results.csv")

## 16. Summary

### Model Architecture:
- **Base Model**: DistilBERT (distilbert-base-uncased)
- **Fine-Tuning Method**: LoRA (Low-Rank Adaptation)
- **LoRA Configuration**: r=8, alpha=16, dropout=0.1
- **Target Modules**: Query and Value attention layers (q_lin, v_lin)
- **Task**: Binary sequence classification (Human vs AI)

### Key Achievements:
- **Parameter Efficiency**: Training only ~0.6% of parameters (LoRA adapters)
- **Memory Efficient**: Adapter weights are only ~1-2MB
- **Performance**: Comparable to full fine-tuning with 99%+ fewer trainable parameters
- **Mimic Analysis**: Evaluated on sophisticated AI texts mimicking specific authors

### Why "The Transformer"?
This model leverages **transformer architecture** with **self-attention**:
- Captures long-range dependencies in text
- Learns contextual relationships between words
- Pre-trained on massive corpora, fine-tuned for stylometry
- LoRA enables efficient adaptation to new tasks

### LoRA Benefits:
1. **Efficiency**: 99%+ reduction in trainable parameters
2. **Speed**: Faster training and lower memory usage
3. **Modularity**: Easy to swap/combine multiple LoRA adapters
4. **Performance**: Maintains accuracy while being parameter-efficient
5. **Deployment**: Tiny adapter files for easy distribution

### Comparison with Other Tiers:
- **Tier B (GloVe + FFN)**: Uses semantic embeddings, simpler architecture
- **Tier C (DistilBERT + LoRA)**: Full transformer with contextual understanding, parameter-efficient
- DistilBERT learns **contextualized representations** (word meaning depends on context)
- GloVe provides **static embeddings** (same vector for word regardless of context)

### Output Files:
- `lora_adapter/` - LoRA weights and tokenizer (~1-2MB)
- `evaluation_results.csv` - Test metrics
- `checkpoint-*/` - Training checkpoints